# 04 - XGBoost 5-Fold CV (GPU)

Third model family, same raw features and the same 5-fold
`StratifiedKFold` split (`random_state=42`) as `02_kfold_cv.ipynb` and
`03_lightgbm_kfold.ipynb`, for the same reason: a fair comparison and
aligned OOF indices for later stacking. This machine has a CUDA GPU
available, so this run uses `device="cuda"` - XGBoost's GPU histogram
method - rather than CPU `hist`.

In [ ]:
import pandas as pd
import numpy as np
import time
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

print("XGBoost ready")

In [ ]:
train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")

TARGET = "Will_Buy_EV"

X = train.drop(columns=[TARGET, "id"])
y = train[TARGET].map({"No": 0, "Yes": 1})
X_test = test.drop(columns=["id"])

cat_cols = X.select_dtypes(include=["object", "category", "str"]).columns.tolist()

# XGBoost's native categorical support (enable_categorical=True below)
# also needs pandas "category" dtype, same requirement as LightGBM.
for c in cat_cols:
    X[c] = X[c].astype("category")
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

print("Train:", X.shape, " Test:", X_test.shape)
print("Categorical columns:", cat_cols)

Train: (668665, 13)  Test: (286571, 13)
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 5-fold CV training (GPU)

`max_depth=6`, `learning_rate=0.05`, up to 3000 trees,
`early_stopping_rounds=50` per fold - deliberately the same
learning-rate scale as the CatBoost/LightGBM runs so this is a
model-family comparison, not a hyperparameter comparison. Optuna tuning
of this config is a separate, later step (Phase 3 backlog).

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_preds = np.zeros((N_SPLITS, len(X_test)))
fold_aucs = []
fold_best_iters = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = XGBClassifier(
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=6,
        tree_method="hist",
        device="cuda",
        enable_categorical=True,
        eval_metric="auc",
        early_stopping_rounds=50,
        random_state=42,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    va_pred = model.predict_proba(X_va)[:, 1]
    oof_pred[va_idx] = va_pred

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_aucs.append(fold_auc)
    fold_best_iters.append(model.best_iteration)
    test_preds[fold] = model.predict_proba(X_test)[:, 1]

    print(f"Fold {fold}: AUC={fold_auc:.6f}  best_iter={model.best_iteration}  elapsed={time.time()-t0:.0f}s")

print("Done in", round(time.time()-t0), "s")

Fold 0: AUC=0.940435  best_iter=437  elapsed=8s
Fold 1: AUC=0.941493  best_iter=378  elapsed=10s
Fold 2: AUC=0.942803  best_iter=483  elapsed=13s
Fold 3: AUC=0.942243  best_iter=433  elapsed=15s
Fold 4: AUC=0.941665  best_iter=436  elapsed=17s
Done in 17 s


> [!info] Device-mismatch warning is a perf note, not a correctness issue
> The model trains on GPU but `predict_proba` receives a plain
> pandas/CPU array, so XGBoost silently falls back to a CPU DMatrix for
> prediction. Predictions are still correct; it's just not as fast as it
> could be. Fix later (not done here) by predicting from a `DMatrix`
> built with the same device, or by keeping data on GPU via a
> `QuantileDMatrix` throughout - worth doing once this becomes the
> model used for heavier Optuna sweeps, not worth the complexity for
> one comparison run.

In [ ]:
oof_auc = roc_auc_score(y, oof_pred)

print("Fold AUCs:", [round(a, 6) for a in fold_aucs])
print("Mean fold AUC:", np.mean(fold_aucs), " Std:", np.std(fold_aucs))
print("OOF AUC:", oof_auc)
print("Best iterations per fold:", fold_best_iters, " mean:", np.mean(fold_best_iters))

Fold AUCs: [0.940435, 0.941493, 0.942803, 0.942243, 0.941665]
Mean fold AUC: 0.9417279712337482  Std: 0.0007939797059047637
OOF AUC: 0.9417202899345948
Best iterations per fold: [437, 378, 483, 433, 436]  mean: 433.4


**XGBoost OOF AUC 0.941720** - ahead of both CatBoost (0.941656, Exp.
002) and LightGBM (0.941470, Exp. 003). This is the first model-family
change in this project that actually improved the score, rather than
just changing training speed. The margin over CatBoost is small
(+0.000064) but the fold pattern is consistent - XGBoost wins or ties
on 4 of 5 folds - so it's a real, if modest, edge, not noise from a
single lucky fold.

In [ ]:
test_pred_avg = test_preds.mean(axis=0)

catboost_sub = pd.read_csv("../submissions/catboost_5fold_baseline.csv")
lightgbm_sub = pd.read_csv("../submissions/lightgbm_5fold_baseline.csv")

corr_cb = np.corrcoef(test_pred_avg, catboost_sub[TARGET].values)[0, 1]
corr_lgb = np.corrcoef(test_pred_avg, lightgbm_sub[TARGET].values)[0, 1]

print("Correlation with CatBoost:", corr_cb)
print("Correlation with LightGBM:", corr_lgb)

Correlation with CatBoost: 0.999021205218093
Correlation with LightGBM: 0.9986395081760944


> [!warning] Still not much ensemble diversity
> Same story as the LightGBM comparison: >0.998 correlation with both
> other models on raw features. All three tree libraries are converging
> on essentially the same function here - expected, since they're
> seeing identical inputs. The diversity this project needs still has
> to come from Phase 2's engineered features (different feature sets
> feeding different models) or a structurally different model (the
> entity-embedding DNN), not from adding a fourth tree library on the
> same 13 raw columns.

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    TARGET: test_pred_avg
})

submission.to_csv("../submissions/xgboost_5fold_baseline.csv", index=False)

print("Submission shape:", submission.shape)
print("Missing values:", submission.isna().sum().sum())
submission.head()

       id  Will_Buy_EV\n0  668665     0.009197\n1  668666     0.019910\n2  668667     0.004361\n3  668668     0.003650\n4  668669     0.019794

## Experiment 004 - XGBoost 5-Fold CV (GPU)

- Features: identical raw features to Experiments 002-003, no engineering
- CV: identical 5-fold `StratifiedKFold` (`random_state=42`) - same fold
  membership as Experiments 002 and 003
- Model: `XGBClassifier`, `max_depth=6`, `learning_rate=0.05`, up to 3000
  trees, `early_stopping_rounds=50` per fold, `device="cuda"`,
  `enable_categorical=True`

**Mean fold AUC: 0.941728 (std 0.000794) - OOF AUC: 0.941720**

| | CatBoost (Exp. 002) | LightGBM (Exp. 003) | XGBoost (Exp. 004) |
|---|---|---|---|
| OOF AUC | 0.941656 | 0.941470 | **0.941720** |
| Public LB | 0.94144 | 0.94124 | pending |
| Training time (5 folds) | ~588s | ~27s | ~17s (GPU) |
| Correlation with CatBoost | - | 0.999 | 0.999 |

**Current best model on raw features: XGBoost.** Small margin (+0.00006
AUC over CatBoost), but consistent across folds, and it trained in 17s
on this machine's GPU versus CatBoost's ~10 minutes on CPU - the fastest
and best-scoring option here, which makes it the natural candidate to
carry into Phase 2's feature-engineering work.

**Next:** submit this to Kaggle to confirm the OOF edge shows up on the
public LB the same way it did for the CatBoost -> LightGBM comparison,
then move to Phase 2 (feature engineering) using XGBoost as the primary
model given its combination of best score and lowest training cost.